Install The SearchLibrium Package


From pypi see resource https://pypi.org/project/SearchLibrium/

In [5]:
!pip install SearchLibrium --upgrade

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Attempting uninstall: SearchLibrium
    Found existing installation: SearchLibrium 0.0.34
    Uninstalling SearchLibrium-0.0.34:
      Successfully uninstalled SearchLibrium-0.0.34


Example of using the SearchLibrium package to estimate a deep nested logit model.
from the resource https://cran.r-project.org/web/packages/mlogit/vignettes/e2nlogit.html

In [1]:
import SearchLibrium as sl

import numpy as np
import pandas as pd



Current version: 0.0.32

              .. .. .. .. .. .. .. ..  .  .  .  .. .. .. .. .. .. .. .. .. ..  .  .  .. .. 
               .. ..  .  .  .. .. .. .. .. .. .. .. .. ..  .  .  .  .. .. .. .. .. .. .. ..
              .............................................................................
              .. .. .. .. .. .. .. ..  .  .  .  .. .. .. .. .. .. .. .. .. ..  .  .  .. .. 
               .. ..  .  .  .. .. .. .. .. .. .. .. .. ..  .  .  .  .. .. .. .. .. .. .. ..
              .. .. .. .. .. .. .. ..  .  .  .  .. .. .. .. .*@:. .. .. .. ..  .  .  .. .. 
              .. .. .. .. .. .. .. ..  .  .  .  .. .. .. .:@@@@@@ .. .. .. ..  .  .  .. .. 
               .. ..  .  .  .. .. .. .. .. .. .. .. .. .@@@@  .+@@  .. .. .. .. .. .. .. ..
              .. .. .. .. .. .. .. ..  .  .  .  .. *@@@@@=. .. .@@=. .. .. ..  .  .  .. .. 
              ...................................@@@@@@%........@@@........................
               .. ..  .  .  .. .. .. .. .. .. ..@@@%@@.

By the way make sure the nests have different names, unformunately this model does not fit, the nest is likely to complex. I am also looking at into putting in bounds, as this will help as well..

In [8]:

def test_deep_nested_probabilities(file ='Heating_Data.csv'):
    # Generate a sample dataset
    df = pd.read_csv(file)
    print(df.head())
    varnames_new = ['ich', 'och','icca','occa']

    # Use the deep nested structure
    nests = {
        "Cooling": {
            "sub_nests": {
                "Gas": {
                    "sub_nests": {
                        "gcc": {"alternatives": [0]},
                        "hpc": {"alternatives": [1]},
                    }
                },
                "Electric": {
                    "sub_nests": {
                        "ecc": {"alternatives": [2]},
                        "erc": {"alternatives": [3]},
                    }
                },            
            }
        },
        "Heating": {
            "sub_nests": {
                "Gas": {"alternatives": [4]},                        
                "Electric": {
                    "sub_nests": {
                        "ec": {"alternatives": [5]},
                        "er": {"alternatives": [6]},
                    }
                },            
            }
        },
    }


        # Test the function
    nests = {
        "Cooling": {
            "sub_nests": {
                "Gas1": {
                    "sub_nests": {
                        "gcc": {"alternatives": [0]},
                        "hpc": {"alternatives": [1]},
                    }
                },
                "Electric": {
                    "sub_nests": {
                        "ecc": {"alternatives": [2]},
                        "erc": {"alternatives": [3]},
                    }
                },            
            }
        },
        "Heating": {
            "sub_nests": {
                "Gas2": {"alternatives": [4]},                        
                "Electric2": {
                    "sub_nests": {
                        "ec": {"alternatives": [5]},
                        "er": {"alternatives": [6]},
                    }
                },            
            }
        },
    }

    

    lambda_mapping = {
        "Cooling": 0,
        "Cooling_Gas": 1,
        "Coooling_Gas_gcc": 2,
        "Cooling_Gas_hpc": 3,
        "Cooling_Electric": 4,  
        "Cooling_Electric_ecc": 5,
        "Cooling_Electric_erc": 6,
        "Heating": 7,
        "Heating_Gas": 8,
        "Heating_Electric": 9,
        "Heating_Electric_ec": 10,
        "Heating_Electric_er": 11 
        
    }

    lambdas = {
            "Cooling": 1,
            "Cooling_Gas": 1,
            "Coooling_Gas_gcc": 1,
            "Cooling_Gas_hpc": 1,
            "Cooling_Electric": 1,  
            "Cooling_Electric_ecc": 1,
            "Cooling_Electric_erc": 1,
            "Heating": 1,
            "Heating_Gas": 1,
            "Heating_Electric": 1,
            "Heating_Electric_ec": 1,
            "Heating_Electric_er": 1
            
        }
    df['avail'] = 1  # Assuming all alternatives are available
    model = sl.multinomial_nested.MultiLayerNestedLogit()
    model.setup(
        X=df[varnames_new],
        y=df['choice'],
        varnames=varnames_new,
        isvars=np.array(['intercept'], dtype=object),
        fit_intercept=True,
        alts=df['alt'],
        ids=df['id'],
        avail=df['avail'],
        nests=nests,
        lambdas=lambdas,
        lambdas_mapping=lambda_mapping,
        return_grad=False
    )
    model.fit()
    model.summarise()
    # Initial beta values (features + lambdas)
    #betas = np.zeros(len(varnames_new) + len(lambdas))
    #probabilities = model.compute_probabilities(betas, model.X, model.avail)

    #print("Probabilities shape:", probabilities.shape)
    #print("Probabilities sum (per observation):", np.sum(probabilities, axis=1))

    ## I took this data and eample from here, just extended to 3 level nest"


path = '../Data/Heating_Data.csv'
test_deep_nested_probabilities(path)

   id  alt    ich   och   icca  occa  income  inc.cooling  inc.room  \
0   1    1   9.70  2.26  27.28  2.95      20           20         0   
1   1    2  11.36  1.73  27.28  2.95      20           20         0   
2   1    3   7.86  4.09  27.28  2.95      20           20         0   
3   1    4   8.79  3.85  27.28  2.95      20           20        20   
4   1    5  24.08  2.26   0.00  0.00      20            0         0   

   int.cooling  choice  
0            1       4  
1            1       4  
2            1       4  
3            1       4  
4            0       4  
nest list ['Cooling', 'Gas1', 'gcc', 'hpc']
nest list ['Cooling', 'Gas1', 'gcc', 'hpc', 'Electric', 'ecc', 'erc']
nest list ['Cooling', 'Gas1', 'gcc', 'hpc', 'Electric', 'ecc', 'erc']
nest list ['Cooling', 'Gas1', 'gcc', 'hpc', 'Electric', 'ecc', 'erc', 'Heating', 'Gas2', 'Electric2', 'ec', 'er']
nest list ['Cooling', 'Gas1', 'gcc', 'hpc', 'Electric', 'ecc', 'erc', 'Heating', 'Gas2', 'Electric2', 'ec', 'er']
nest list [

In [18]:
#test code
lambda_mapping = {
    "Cooling": 0,
    "Cooling_Gas": 1,
    "Cooling_Gas_gcc": 2,
    "Cooling_Gas_hpc": 3,
    "Cooling_Electric": 4,  
    "Cooling_Electric_ecc": 5,
    "Cooling_Electric_erc": 6,
    "Heating": 7,
    "Heating_Gas": 8,
    "Heating_Electric": 9,
    "Heating_Electric_ec": 10,
    "Heating_Electric_er": 11
}

lambdas = {
    "Cooling": 1,
    "Cooling_Gas": 1,
    "Cooling_Gas_gcc": 1,
    "Cooling_Gas_hpc": 1,
    "Cooling_Electric": 1,  
    "Cooling_Electric_ecc": 1,
    "Cooling_Electric_erc": 1,
    "Heating": 1,
    "Heating_Gas": 1,
    "Heating_Electric": 1,
    "Heating_Electric_ec": 1,
    "Heating_Electric_er": 1
}

nests = {
    "Cooling": {
        "sub_nests": {
            "Gas": {
                "sub_nests": {
                    "gcc": {"alternatives": [0]},
                    "hpc": {"alternatives": [1]},
                }
            },
            "Electric": {
                "sub_nests": {
                    "ecc": {"alternatives": [2]},
                    "erc": {"alternatives": [3]},
                }
            },            
        }
    },
    "Heating": {
        "sub_nests": {
            "Gas": {"alternatives": [4]},                        
            "Electric": {
                "sub_nests": {
                    "ec": {"alternatives": [5]},
                    "er": {"alternatives": [6]},
                }
            },            
        }
    },
}
lambda_mapping = {}
index = 0
lambdas = {}

# Recursive function to populate lambda_mapping and lambdas
def assign_lambdas(nests, prefix=""):
    global index
    for nest_name, nest_info in nests.items():
        full_name = f"{prefix}{nest_name}" if prefix else nest_name
        lambda_mapping[full_name] = index
        lambdas[full_name] = 1  # Initialize lambda to 1
        index += 1
        if "sub_nests" in nest_info:
            assign_lambdas(nest_info["sub_nests"], prefix=f"{full_name}_")

# Populate lambda_mapping and lambdas
assign_lambdas(nests)

# Print results to validate
print("Lambda Mapping:", lambda_mapping)
print("Lambdas:", lambdas)

Lambda Mapping: {'Cooling': 0, 'Cooling_Gas': 1, 'Cooling_Gas_gcc': 2, 'Cooling_Gas_hpc': 3, 'Cooling_Electric': 4, 'Cooling_Electric_ecc': 5, 'Cooling_Electric_erc': 6, 'Heating': 7, 'Heating_Gas': 8, 'Heating_Electric': 9, 'Heating_Electric_ec': 10, 'Heating_Electric_er': 11}
Lambdas: {'Cooling': 1, 'Cooling_Gas': 1, 'Cooling_Gas_gcc': 1, 'Cooling_Gas_hpc': 1, 'Cooling_Electric': 1, 'Cooling_Electric_ecc': 1, 'Cooling_Electric_erc': 1, 'Heating': 1, 'Heating_Gas': 1, 'Heating_Electric': 1, 'Heating_Electric_ec': 1, 'Heating_Electric_er': 1}


Lambda Mapping: {'Cooling': 0, 'Gas': 8, 'gcc': 2, 'hpc': 3, 'Electric': 9, 'ecc': 5, 'erc': 6, 'Heating': 7, 'ec': 10, 'er': 11}
Nest List: ['Cooling', 'Gas', 'gcc', 'hpc', 'Electric', 'ecc', 'erc', 'Heating', 'Gas', 'Electric', 'ec', 'er']
length of nest_list: 12
